# 02 · Slice-count search, final model and evaluation

Runs steps 3–5 of the pipeline on the cached DINOv2 features and looks at the results.
Only the small attention head is trained here, so a CPU is enough (a few minutes).

**Prerequisite:** a feature cache in `features/`, either from `python extract_features.py`
(needs the MRI and a GPU) or copied from the original Kaggle run with
`python import_kaggle_run.py idh_dinov2_auto_slices_v1.zip --features`.

The same steps are available as scripts: `search_slices.py`, `train.py`, `evaluate.py`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # make the `idh` package importable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from idh.config import SearchConfig
from idh.features import load_features, load_manifest, make_bags
from idh.metrics import age_baseline_scores, bootstrap_auroc_ci, choose_threshold, summarize
from idh.plots import plot_slice_search, plot_test_results
from idh.selection import search_slice_counts
from idh.training import fit_head, predict_proba
from idh.utils import get_device

FEATURES = os.environ.get("IDH_FEATURES", "../features")
settings = SearchConfig()
device = get_device()

manifest = load_manifest(FEATURES)
parts = {s: manifest[manifest["split"] == s].reset_index(drop=True) for s in ("train", "validation", "test")}
store = load_features(FEATURES, manifest, settings.feature_kind)
print({s: len(f) for s, f in parts.items()}, "| slices per patient:",
      int(manifest["n_slices"].min()), "-", int(manifest["n_slices"].max()))

## Step 3 · How many slices per patient?

Each candidate count k samples k evenly spaced slices from the central 90% of the brain
extent. Each candidate is cross-validated on the **training patients only** with identical
folds and seeds. Within 0.005 AUROC of the best, the smaller k wins.

In [ ]:
selection, results, fold_results, oof = search_slice_counts(parts["train"], store, settings, device)
plot_slice_search(results, selection["selected_candidate"])
results.sort_values("mean_cv_auroc", ascending=False).round(4)

The fold standard deviations (0.02–0.11) are as large as most differences between candidates,
so anything from 16 to 64 slices is roughly equivalent. Only very small counts (4–12) are
clearly worse. `--cv-folds 5 --cv-repeats 3` in `search_slices.py` gives a less noisy ranking.

In [ ]:
selection

## Step 4 · Final model and threshold

A fresh head is trained on all 345 training patients for the CV-derived number of epochs.
The 75 validation patients only choose the decision threshold (maximum balanced accuracy).

In [ ]:
count = selection["selected_count"]
bags = make_bags(manifest["patient_id"], store, count, settings.lower_fraction, settings.upper_fraction)
model, history, _ = fit_head(parts["train"], bags, settings, settings.seed + 10000, device,
                             epochs=selection["final_training_epochs"])

val_scores = predict_proba(model, parts["validation"], bags, device)
threshold, threshold_curve = choose_threshold(parts["validation"]["label"], val_scores)
print(f"Threshold: {threshold:.3f}")
pd.Series(summarize(parts["validation"]["label"], val_scores, threshold)).to_frame("validation")

## Step 5 · Test set (used once)

In [ ]:
test = parts["test"]
labels = test["label"].to_numpy()
scores = predict_proba(model, test, bags, device)

age_val, _ = age_baseline_scores(parts["train"], parts["validation"])
age_threshold, _ = choose_threshold(parts["validation"]["label"], age_val)
age_test, _ = age_baseline_scores(parts["train"], test)

table = pd.DataFrame({
    "DINOv2 + attention MIL": summarize(labels, scores, threshold),
    "Age only": summarize(labels, age_test, age_threshold),
    "Always wildtype": summarize(labels, np.full(len(test), parts["train"]["label"].mean()), 0.5),
})
print("Image model AUROC 95% CI:", np.round(bootstrap_auroc_ci(labels, scores, 1000, settings.seed + 50000), 3))
table.loc[["auroc", "accuracy", "balanced_accuracy", "sensitivity", "specificity", "ppv", "npv", "threshold"]].astype(float).round(3)

In [ ]:
plot_test_results(labels, {"DINOv2 + MIL": scores, "Age only": age_test}, threshold,
                  (scores >= threshold).astype(int));

## Where does the head look?

Attention weights over the 64 slices of the most confidently predicted mutant test patient.
Attention shows what the pooling emphasised; it is not a tumour localisation.

In [ ]:
import torch
from idh.training import collate_bags

pid = test.loc[np.argmax(scores), "patient_id"]
x, mask, _ = collate_bags([(torch.from_numpy(bags[pid]), 0.0)])
with torch.inference_mode():
    _, attention = model(x.to(device), mask.to(device), return_attention=True)
_, fractions, z_mm = store[pid]

from idh.features import select_slices
chosen = select_slices(fractions, count, settings.lower_fraction, settings.upper_fraction)
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.bar(z_mm[chosen], attention[0].cpu().numpy(), width=1.5, color="#cf734b")
ax.set(xlabel="Axial position (mm)", ylabel="Attention", title=f"{pid}: score {scores.max():.2f}")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()